# 01 工作流编排：统一 YAML 之问与 Temporal 的定位

核心问题：能不能写**一份**工作流文档，交给各种引擎（Kestra / Windmill / Argo / Temporal…）解析？先看格局图：`../images/fig3_workflow_landscape.png`。

## 实验 1：一份文档真实跑起来

`workflow_dsl_demo.py` 内置一份 JSON 文档（JSON ⊂ YAML）。mini 引擎做拓扑排序、模板展开、审批门，并把运行账本原子写入磁盘。

In [1]:

import subprocess, sys
from pathlib import Path

SCRIPTS = Path.cwd().parent / "scripts"

def run(script, *args):
    proc = subprocess.run(
        [sys.executable, str(SCRIPTS / script), *args],
        capture_output=True, text=True, timeout=120,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc.returncode

run("workflow_dsl_demo.py", "--auto-approve", "--workdir", "/tmp/orch-nb-wfdemo")

# executing workflow 'nightly-report'  run=run-40c3a58d

  [step_started] {'step': 'collect', 'type': 'agent'}
  [step_completed] {'step': 'collect', 'output': "[researcher] done: Collect yesterday's ops incidents"}
  [step_started] {'step': 'draft', 'type': 'agent'}
  [step_completed] {'step': 'draft', 'output': '[writer] done: Write a summary of: [researcher] done: Collect y'}
  [approval_requested] {'step': 'approve', 'approvers': ['oncall-human']}
  [approval_resolved] {'step': 'approve', 'decision': 'approved'}
  [step_started] {'step': 'send', 'type': 'agent'}
  [step_completed] {'step': 'send', 'output': '[publisher] done: Publish the approved report'}

# run completed; ledger -> /tmp/orch-nb-wfdemo/runs/run-40c3a58d.json



0

看看刚写下的运行账本——这就是任何编排引擎都必须提供的最小持久化面：

In [2]:
import json
from pathlib import Path

ledgers = sorted(Path('/tmp/orch-nb-wfdemo/runs').glob('*.json'))
latest = json.loads(ledgers[-1].read_text())
print('run:', latest['run_id'], '->', latest['status'])
for ev in latest['events']:
    print(f"  [{ev['event']}]", {k: v for k, v in ev.items() if k not in ('t', 'event')})

run: run-fd1f14a2 -> completed
  [step_started] {'step': 'collect', 'agent': 'researcher', 'prompt': "Collect yesterday's ops incidents"}
  [step_completed] {'step': 'collect', 'output': "[researcher] done: Collect yesterday's ops incidents"}
  [step_started] {'step': 'draft', 'agent': 'writer', 'prompt': "Write a summary of: [researcher] done: Collect yesterday's ops incidents"}
  [step_completed] {'step': 'draft', 'output': '[writer] done: Write a summary of: [researcher] done: Collect y'}
  [approval_requested] {'step': 'approve', 'approvers': ['oncall-human']}
  [approval_resolved] {'step': 'approve', 'decision': 'approved'}
  [step_started] {'step': 'send', 'agent': 'publisher', 'prompt': 'Publish the approved report'}
  [step_completed] {'step': 'send', 'output': '[publisher] done: Publish the approved report'}


## 实验 2：同一份文档，三种引擎语法

`--show-ports` 把同一份文档机械翻译成 Kestra YAML / Windmill OpenFlow / Argo CRD。观察哪些字段能直译、哪些只能注释（那就是'共同分母'的边界）。

In [3]:

import subprocess, sys
from pathlib import Path

SCRIPTS = Path.cwd().parent / "scripts"

def run(script, *args):
    proc = subprocess.run(
        [sys.executable, str(SCRIPTS / script), *args],
        capture_output=True, text=True, timeout=120,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc.returncode

run("workflow_dsl_demo.py", "--show-ports")

== Same document, three engine ports (teaching approximations) ==

--- Kestra (YAML, everything-is-YAML) ---
id: nightly-report
namespace: demo
tasks:
  - id: collect
    type: io.kestra.plugin.fs.http.Request
    # -> agent/sandbox endpoint for step collect
  - id: draft
    type: io.kestra.plugin.fs.http.Request
    # -> agent/sandbox endpoint for step draft
    dependsOn: [collect]
  - id: approve
    type: io.kestra.plugin.core.flow.Pause
    dependsOn: [draft]
  - id: send
    type: io.kestra.plugin.fs.http.Request
    # -> agent/sandbox endpoint for step send
    dependsOn: [approve]
triggers:
  - id: cron
    type: io.kestra.plugin.core.trigger.Schedule
    cron: 0 3 * * *

--- Windmill OpenFlow (flow.yaml) ---
summary: port of nightly-report (OpenFlow schema, simplified)
value:
  modules:
    - type: rawscript
      id: collect
      # module calls agent/sandbox HTTP endpoint for step collect
    - type: rawscript
      id: draft
      # module calls agent/sandbox HTTP endpoint

0

## Temporal 对照：workflow-as-code，没有官方 YAML

要点（均已核实）：durable execution = 代码从事件历史重放；审批 = Signal、状态 = Query；服务端 MIT，PostgreSQL + 2GB 即可自托管，`temporal server start-dev` 本地即起；2026 年已 GA 的 agent 集成：OpenAI Agents SDK（2026-03）、LangGraph 插件（2026-07）、ADK。

下面读一下参考实现的关键段落：

In [4]:
from pathlib import Path

src = (Path.cwd().parent / 'scripts' / 'temporal_agent_workflow.py').read_text()
lines = src.splitlines()
print('\n'.join(lines[32:78]))

        "temporalio is not installed.\n"
        "  pip install temporalio\n"
        "  temporal server start-dev   # from https://docs.temporal.io/cli\n"
        "Then re-run: python temporal_agent_workflow.py [worker|start]"
    )


@dataclass
class Approval:
    approved: bool
    approver: str


@activity.defn
async def call_llm(prompt: str) -> str:
    """Stand-in for the real LLM/agent call. Activities (not workflows) are
    where side effects and non-determinism live — they get automatic retries."""
    return f"[toy-llm] draft answer for: {prompt[:40]}"


@workflow.defn
class AgentApprovalWorkflow:
    """Agent step -> wait for a human approval signal -> finalize.

    Crash-safe by construction: if the worker dies while waiting, Temporal
    re-schedules the workflow and it resumes at wait_condition with the
    signal's event already journaled.
    """

    def __init__(self) -> None:
        self.approval: Approval | None = None
        self.draft: str = ""

    @workflow.

## 实验 3：真实形态 —— incident-response（同一份工作流，六种写法）

内置文档 `incident-response` 贴近 cloud-agent 设计：SQL 取视图 → broker 起沙箱（`tier` 声明隔离档位）→ 两个 agent prompt → 审批 → 发布并回收。

`examples/incident-response/` 目录里有同一份工作流的 Kestra / Windmill OpenFlow / Argo / GitHub Actions 手写版本，逐元素对照表见该目录 README——重点观察 retry/approval/表达式三种“方言”。

In [5]:

import subprocess, sys
from pathlib import Path

SCRIPTS = Path.cwd().parent / "scripts"

def run(script, *args):
    proc = subprocess.run(
        [sys.executable, str(SCRIPTS / script), *args],
        capture_output=True, text=True, timeout=120,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc.returncode

run("workflow_dsl_demo.py", "--workflow", "incident-response", "--auto-approve", "--workdir", "/tmp/orch-nb-incident")

# executing workflow 'incident-response'  run=run-782023eb

  [step_started] {'step': 'fetch-view', 'type': 'sql'}
  → [sql/postgres] SELECT * FROM v_daily_incidents WHERE day = '${{ trigger.date }}'
  ← 2 rows (toy result)
  [step_completed] {'step': 'fetch-view', 'output': '[{"id": 101, "svc": "billing", "level": "P1"}, {"id": 102, "svc": "search", "lev'}
  [step_started] {'step': 'provision-env', 'type': 'provision'}
  → POST http://sandbox-broker:8080/sandboxes {image: ops-agent-base:3, tier: container, ttl: 1800s, egress: {'pypi': True, 'npm': False}}
  ← 201 {'sandbox_id': 'sbx-4152c7', 'endpoint': 'http://10.0.0.23:4123', 'tier': 'container'}   (到期自动暂停；broker 决定 tier 落到哪个 runner)
  [step_completed] {'step': 'provision-env', 'output': '{"sandbox_id": "sbx-4152c7", "endpoint": "http://10.0.0.23:4123", "tier": "conta'}
  [step_started] {'step': 'agent-triage', 'type': 'agent'}
  [step_completed] {'step': 'agent-triage', 'output': '[triage-bot] done: 事件初判严重级别（P0-P3）：[{"id": 101, "sv

0

In [6]:
from pathlib import Path

ex = Path.cwd().parent / 'examples' / 'incident-response'
for f in sorted(ex.iterdir()):
    print(f.name)

README.md
argo.yaml
github-actions.yml
kestra.yaml
windmill.flow.yaml
workflow.json
workflow.yml


**检查点**：
1. 你的 Kestra 翻译里，哪一步丢失了原文的 `retries` 语义？
2. 为什么 Temporal 把审批建模成 signal 而不是轮询数据库？
3. 对'短 DAG + 审批'的场景，列出选自研小执行器 vs Temporal 的各两条理由。